El notebook tiene como finalidad evaluar el procesamiento de diferentes volúmenes de datos sintéticos mediante Apache Spark en Databricks, utilizando información de ventas y detalles de ventas de una librería escolar.
se trabajara con 3 escenarios:
- BD01:10 000 VENTAS
- BD02:50 000 VENTAS
- BD03: 100 000 VENTAS
##En cada escenario se realizan operaciones de lectura, unión, transformación, filtrado y agregación para obtener indicadores comerciales y medir el tiempo de procesamiento.

In [0]:
##CONFIGURACION DE LA RUTA DE ALMACENAMIENTO
#Define la ubicación principal donde se almacenan los archivos Parquet de los tres escenarios de prueba.
BASE_PATH = "/Volumes/solucion_bd/datos_bd/datos_libreria"

print("Ruta configurada:", BASE_PATH)

Ruta configurada: /Volumes/solucion_bd/datos_bd/datos_libreria


In [0]:
##VERIFICACION DE LOS ARCHIVOS ALMACENADOS
##listar los archivos y carpetas existentes en la ubicación configurada
display(
    dbutils.fs.ls(BASE_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/,BD01/,0,1795671460164
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD02/,BD02/,0,1795671460164
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD03/,BD03/,0,1795671460164


In [0]:
##Lee los archivos Parquet correspondientes a las ventas y sus detalles mediante spark.read.parquet
##prepara los conjuntos de datos para realizar operaciones de procesamiento distribuido con Spark.
ruta_ventas = f"{BASE_PATH}/BD01/venta_sintetica.parquet"
df_ventas = spark.read.parquet(ruta_ventas)

display(df_ventas.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado
108,4,11,4,2025-09-10 00:05:17,CANCELADO
109,10,4,3,2026-03-15 07:16:31,COMPLETADO
110,1,3,1,2026-07-13 16:12:26,COMPLETADO
111,9,3,2,2026-10-27 12:29:11,COMPLETADO
112,6,4,4,2026-11-09 13:21:44,COMPLETADO
113,10,12,4,2026-06-19 23:16:47,COMPLETADO
114,2,1,5,2026-09-15 08:38:51,COMPLETADO
115,9,12,5,2026-10-24 20:15:08,COMPLETADO
116,5,11,1,2026-05-02 05:43:16,PENDIENTE
117,4,2,3,2026-01-09 07:10:56,COMPLETADO


In [0]:
print("Total de ventas:", df_ventas.count())

Total de ventas: 10000


In [0]:
##Lee los archivos Parquet correspondientes a las ventas y sus detalles mediante spark.read.parquet
##prepara los conjuntos de datos para realizar operaciones de procesamiento distribuido con Spark.
ruta_detalles = f"{BASE_PATH}/BD01/ventaDetalle_sintetico.parquet"
df_detalles = spark.read.parquet(ruta_detalles)

display(df_detalles.limit(10))

print("Total de detalles:", df_detalles.count())

idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
183,108,7,5,17.0,12.0
184,108,9,5,28.0,18.0
185,109,3,1,1.5,0.8
186,109,21,4,13.0,7.5
187,109,2,1,1.5,0.8
188,110,7,1,17.0,12.0
189,110,22,4,7.5,4.0
190,110,16,3,10.5,6.0
191,111,21,5,13.0,7.5
192,111,11,2,55.0,35.0


Total de detalles: 29775


El método .count() calcula la cantidad de registros existentes en cada DataFrame.
## el primer caso se obtuvo 10 000 registros de ventas y 29 775 registros de detalles

In [0]:
for escenario in ["BD01", "BD02", "BD03"]:

    ventas = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/venta_sintetica.parquet"
    )

    detalles = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/ventaDetalle_sintetico.parquet"
    )

    print(f"\n--- {escenario} ---")
    print("Ventas:", ventas.count())
    print("Detalles:", detalles.count())


--- BD01 ---
Ventas: 10000
Detalles: 29775

--- BD02 ---
Ventas: 50000
Detalles: 150069

--- BD03 ---
Ventas: 100000
Detalles: 300003


In [0]:
display(
    dbutils.fs.ls(f"{BASE_PATH}/BD01/")
)

path,name,size,modificationTime
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/ventaDetalle_sintetico.parquet,ventaDetalle_sintetico.parquet,333866,1795195100000
dbfs:/Volumes/solucion_bd/datos_bd/datos_libreria/BD01/venta_sintetica.parquet,venta_sintetica.parquet,194024,1795195100000


In [0]:
for escenario in ["BD01", "BD02", "BD03"]:

    print(f"\nContenido de {escenario}:")

    archivos = dbutils.fs.ls(
        f"{BASE_PATH}/{escenario}/"
    )

    for archivo in archivos:
        print(archivo.name, archivo.size, "bytes")


Contenido de BD01:
ventaDetalle_sintetico.parquet 333866 bytes
venta_sintetica.parquet 194024 bytes

Contenido de BD02:
ventaDetalle_sintetico.parquet 1642131 bytes
venta_sintetica.parquet 964816 bytes

Contenido de BD03:
ventaDetalle_sintetico.parquet 3076088 bytes
venta_sintetica.parquet 1841579 bytes


In [0]:
df_ventas = spark.read.parquet(
    f"{BASE_PATH}/BD01/venta_sintetica.parquet"
)

df_detalles = spark.read.parquet(
    f"{BASE_PATH}/BD01/ventaDetalle_sintetico.parquet"
)

print("Ventas:", df_ventas.count())
print("Detalles:", df_detalles.count())

Ventas: 10000
Detalles: 29775


In [0]:
display(df_ventas.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado
108,4,11,4,2025-09-10 00:05:17,CANCELADO
109,10,4,3,2026-03-15 07:16:31,COMPLETADO
110,1,3,1,2026-07-13 16:12:26,COMPLETADO
111,9,3,2,2026-10-27 12:29:11,COMPLETADO
112,6,4,4,2026-11-09 13:21:44,COMPLETADO
113,10,12,4,2026-06-19 23:16:47,COMPLETADO
114,2,1,5,2026-09-15 08:38:51,COMPLETADO
115,9,12,5,2026-10-24 20:15:08,COMPLETADO
116,5,11,1,2026-05-02 05:43:16,PENDIENTE
117,4,2,3,2026-01-09 07:10:56,COMPLETADO


In [0]:
display(df_detalles.limit(10))

idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
183,108,7,5,17.0,12.0
184,108,9,5,28.0,18.0
185,109,3,1,1.5,0.8
186,109,21,4,13.0,7.5
187,109,2,1,1.5,0.8
188,110,7,1,17.0,12.0
189,110,22,4,7.5,4.0
190,110,16,3,10.5,6.0
191,111,21,5,13.0,7.5
192,111,11,2,55.0,35.0


ANALISIS Y PROCESAMIENTO

In [0]:
##El método .printSchema() muestra los nombres de las columnas y sus respectivos tipos de datos.

print("COLUMNAS DE VENTAS")
df_ventas.printSchema()

print("\nCOLUMNAS DE DETALLES")
df_detalles.printSchema()

COLUMNAS DE VENTAS
root
 |-- idVenta: long (nullable = true)
 |-- fkCliente: long (nullable = true)
 |-- fkUsuario: long (nullable = true)
 |-- fkMetodoPago: long (nullable = true)
 |-- fecha: string (nullable = true)
 |-- estado: string (nullable = true)


COLUMNAS DE DETALLES
root
 |-- idVentaDetalle: long (nullable = true)
 |-- fkVenta: long (nullable = true)
 |-- fkProducto: long (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precioVenta: double (nullable = true)
 |-- costoUnitario: double (nullable = true)



In [0]:

print("Total de ventas:", df_ventas.count())

print("Total de detalles:", df_detalles.count())

Total de ventas: 10000
Total de detalles: 29775


In [0]:
display(
    df_ventas.groupBy("estado")
    .count()
    .orderBy("count", ascending=False)
)

estado,count
COMPLETADO,8801
CANCELADO,737
PENDIENTE,462


In [0]:
display(
    df_detalles.groupBy()
    .count()
)

count
29775


PRUEBAS

In [0]:
display(
    df_ventas
    .groupBy("estado")
    .count()
    .orderBy("estado")
)

estado,count
CANCELADO,737
COMPLETADO,8801
PENDIENTE,462


In [0]:
## Unión de ventas y detalles
#Realiza una unión entre los DataFrames de ventas y detalles utilizando el identificador de venta.
#Propósito: integrar la información de las ventas con sus respectivos productos y detalles para realizar el análisis comercial.

df_unificado = (
    df_ventas.alias("v")
    .join(
        df_detalles.alias("d"),
        df_ventas.idVenta == df_detalles.fkVenta,
        "inner"
    )
)

display(df_unificado.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado,idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario
108,4,11,4,2025-09-10 00:05:17,CANCELADO,183,108,7,5,17.0,12.0
108,4,11,4,2025-09-10 00:05:17,CANCELADO,184,108,9,5,28.0,18.0
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,185,109,3,1,1.5,0.8
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,186,109,21,4,13.0,7.5
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,187,109,2,1,1.5,0.8
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,188,110,7,1,17.0,12.0
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,189,110,22,4,7.5,4.0
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,190,110,16,3,10.5,6.0
111,9,3,2,2026-10-27 12:29:11,COMPLETADO,191,111,21,5,13.0,7.5
111,9,3,2,2026-10-27 12:29:11,COMPLETADO,192,111,11,2,55.0,35.0


In [0]:
##Transformación y cálculo de indicadores
#Calcula tres nuevos campos mediante operaciones aritméticas sobre las columnas existentes.
#Propósito: obtener medidas económicas que permitan analizar el comportamiento de las ventas y la utilidad generada.
from pyspark.sql.functions import col

df_analisis = (
    df_unificado
    .withColumn(
        "importeVenta",
        col("cantidad") * col("precioVenta")
    )
    .withColumn(
        "costoTotal",
        col("cantidad") * col("costoUnitario")
    )
    .withColumn(
        "utilidad",
        col("importeVenta") - col("costoTotal")
    )
)

display(df_analisis.limit(10))

idVenta,fkCliente,fkUsuario,fkMetodoPago,fecha,estado,idVentaDetalle,fkVenta,fkProducto,cantidad,precioVenta,costoUnitario,importeVenta,costoTotal,utilidad
108,4,11,4,2025-09-10 00:05:17,CANCELADO,183,108,7,5,17.0,12.0,85.0,60.0,25.0
108,4,11,4,2025-09-10 00:05:17,CANCELADO,184,108,9,5,28.0,18.0,140.0,90.0,50.0
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,185,109,3,1,1.5,0.8,1.5,0.8,0.7
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,186,109,21,4,13.0,7.5,52.0,30.0,22.0
109,10,4,3,2026-03-15 07:16:31,COMPLETADO,187,109,2,1,1.5,0.8,1.5,0.8,0.7
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,188,110,7,1,17.0,12.0,17.0,12.0,5.0
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,189,110,22,4,7.5,4.0,30.0,16.0,14.0
110,1,3,1,2026-07-13 16:12:26,COMPLETADO,190,110,16,3,10.5,6.0,31.5,18.0,13.5
111,9,3,2,2026-10-27 12:29:11,COMPLETADO,191,111,21,5,13.0,7.5,65.0,37.5,27.5
111,9,3,2,2026-10-27 12:29:11,COMPLETADO,192,111,11,2,55.0,35.0,110.0,70.0,40.0


In [0]:
df_completadas = df_analisis.filter(
    col("estado") == "COMPLETADO"
)

display(df_completadas.agg(
    {"importeVenta": "sum",
     "costoTotal": "sum",
     "utilidad": "sum",
     "cantidad": "sum"}
))

sum(importeVenta),sum(costoTotal),sum(utilidad),sum(cantidad)
1457894.6999999995,861823.6000000017,596071.0999999981,78279


In [0]:
## Análisis de las ventas por estado
#Agrupa los registros según el estado de la venta y calcula los importes acumulados.
#Propósito: observar el comportamiento económico de los diferentes estados de las transacciones.
display(
    df_analisis
    .groupBy("estado")
    .sum(
        "importeVenta",
        "costoTotal",
        "utilidad",
        "cantidad"
    )
)

estado,sum(importeVenta),sum(costoTotal),sum(utilidad),sum(cantidad)
COMPLETADO,1457894.6999999995,861823.6000000017,596071.0999999981,78279
PENDIENTE,73021.79999999999,43218.799999999996,29803.000000000015,4168
CANCELADO,114886.29999999997,67628.09999999998,47258.19999999995,6608


calcular indicadores

In [0]:
##Filtrado de ventas completadas y cálculo de indicadores
#Selecciona exclusivamente las ventas con estado COMPLETADO y calcula los indicadores comerciales.
#Propósito: obtener indicadores comerciales válidos para el análisis, excluyendo las transacciones pendientes y canceladas.
from pyspark.sql.functions import sum, countDistinct, round

indicadores_BD01 = df_analisis.filter(
    col("estado") == "COMPLETADO"
).agg(
    countDistinct("idVenta").alias("Total_Ventas"),
    sum("cantidad").alias("Unidades_Vendidas"),
    round(sum("importeVenta"), 2).alias("Ventas_Totales"),
    round(sum("costoTotal"), 2).alias("Costo_Total"),
    round(sum("utilidad"), 2).alias("Utilidad_Total")
)

display(indicadores_BD01)

Total_Ventas,Unidades_Vendidas,Ventas_Totales,Costo_Total,Utilidad_Total
8801,78279,1457894.7,861823.6,596071.1


In [0]:
import time

In [0]:
##Comparación de los escenarios BD01, BD02 y BD03
#import time: permite medir el tiempo de ejecución.
#import math: proporciona funciones matemáticas.
#resultados = []: crea una lista para almacenar los resultados.
#for escenario in [...]: recorre automáticamente los tres escenarios.
#Propósito: ejecutar el mismo procedimiento de análisis sobre diferentes volúmenes de información.


import time
import math
from pyspark.sql.functions import col, sum

resultados = []

for escenario in ["BD01", "BD02", "BD03"]:

    # Leer archivos
    ventas = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/venta_sintetica.parquet"
    )

    detalles = spark.read.parquet(
        f"{BASE_PATH}/{escenario}/ventaDetalle_sintetico.parquet"
    )

    ##Inicio de la medición
    #Registra el instante inicial de la medición mediante el reloj de Python.
    inicio = time.time()

    # Unir tablas
    unificado = ventas.join(
        detalles,
        ventas.idVenta == detalles.fkVenta,
        "inner"
    )

    # Calcular importes y utilidad
    analisis = (
        unificado
        .withColumn(
            "importeVenta",
            col("cantidad") * col("precioVenta")
        )
        .withColumn(
            "costoTotal",
            col("cantidad") * col("costoUnitario")
        )
        .withColumn(
            "utilidad",
            col("importeVenta") - col("costoTotal")
        )
    )

    # Filtrar ventas completadas
    completadas = analisis.filter(
        col("estado") == "COMPLETADO"
    )

    # Calcular indicadores
    indicadores = completadas.agg(
        sum("idVenta").alias("suma_id"),
        sum("cantidad").alias("unidades"),
        sum("importeVenta").alias("importe"),
        sum("costoTotal").alias("costo"),
        sum("utilidad").alias("utilidad")
    ).first()

    total_ventas = completadas.select("idVenta").distinct().count()
    total_unidades = indicadores["unidades"] or 0
    total_importe = indicadores["importe"] or 0
    total_costo = indicadores["costo"] or 0
    total_utilidad = indicadores["utilidad"] or 0

    fin = time.time()
    tiempo = math.floor((fin - inicio) * 10000) / 10000

    resultados.append((
        escenario,
        ventas.count(),
        detalles.count(),
        total_ventas,
        total_unidades,
        float(total_importe),
        float(total_costo),
        float(total_utilidad),
        tiempo
    ))

# Crear tabla comparativa
columnas = [
    "Escenario",
    "Ventas",
    "Detalles",
    "Ventas_Completadas",
    "Unidades_Vendidas",
    "Importe_Total",
    "Costo_Total",
    "Utilidad_Total",
    "Tiempo_Procesamiento"
]

df_resultados = spark.createDataFrame(resultados, columnas)

display(df_resultados)

Escenario,Ventas,Detalles,Ventas_Completadas,Unidades_Vendidas,Importe_Total,Costo_Total,Utilidad_Total,Tiempo_Procesamiento
BD01,10000,29775,8801,78279,1457894.6999999995,861823.6000000017,596071.0999999981,2.0661
BD02,50000,150069,44066,396560,7324825.099999989,4326778.299999995,2998046.799999985,1.9482
BD03,100000,300003,88080,794129,1.4701315499999998E7,8689102.60000008,6012212.899999929,2.1078


Databricks visualization. Run in Databricks to view.